In [ ]:
"""
=============================================================
FILE 34 — HIERARCHICAL AGENTS
=============================================================

CONCEPTS TAUGHT
----------------
1. Hierarchical Agents
2. Manager Agents
3. Worker Agents
4. Enterprise AI Structures
5. Delegation
6. Multi-Level Orchestration
7. Agent Supervision
8. Specialized AI Teams
9. Distributed Intelligence
10. AI Organization Structures

CORE IDEA
-----------
Higher-level agents manage lower-level agents.

FLOW
-----
Manager Agent
   ↓
Specialized Worker Agents
   ↓
Aggregation

REAL WORLD USE CASES
---------------------
- Enterprise AI systems
- Autonomous organizations
- Multi-team AI workflows
- AI operations platforms
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — STATE
# ============================================================

class State(TypedDict):
    business_problem: str

    finance_team_output: str
    marketing_team_output: str
    operations_team_output: str

    final_strategy: str

# ============================================================
# STEP 5 — WORKER AGENTS
# ============================================================

def finance_agent(state: State):

    response = llm.invoke(
        f"""
        Solve this business problem
        from FINANCE perspective.

        {state['business_problem']}
        """
    )

    return {
        "finance_team_output": response.content
    }

def marketing_agent(state: State):

    response = llm.invoke(
        f"""
        Solve this business problem
        from MARKETING perspective.

        {state['business_problem']}
        """
    )

    return {
        "marketing_team_output": response.content
    }

def operations_agent(state: State):

    response = llm.invoke(
        f"""
        Solve this business problem
        from OPERATIONS perspective.

        {state['business_problem']}
        """
    )

    return {
        "operations_team_output": response.content
    }

# ============================================================
# STEP 6 — MANAGER AGENT
# ============================================================

def manager_agent(state: State):

    response = llm.invoke(
        f"""
        Combine all departmental recommendations.

        FINANCE:
        {state['finance_team_output']}

        MARKETING:
        {state['marketing_team_output']}

        OPERATIONS:
        {state['operations_team_output']}

        Create one enterprise strategy.
        """
    )

    return {
        "final_strategy": response.content
    }

# ============================================================
# STEP 7 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node("finance_agent", finance_agent)
builder.add_node("marketing_agent", marketing_agent)
builder.add_node("operations_agent", operations_agent)

builder.add_node("manager_agent", manager_agent)

# ============================================================
# STEP 8 — PARALLEL EXECUTION
# ============================================================

builder.add_edge(START, "finance_agent")
builder.add_edge(START, "marketing_agent")
builder.add_edge(START, "operations_agent")

builder.add_edge("finance_agent", "manager_agent")
builder.add_edge("marketing_agent", "manager_agent")
builder.add_edge("operations_agent", "manager_agent")

builder.add_edge("manager_agent", END)

# ============================================================
# STEP 9 — COMPILE GRAPH
# ============================================================

graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

# ============================================================
# STEP 10 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "business_problem":
        """
        AI adoption strategy for a retail company
        """
    }
)

# ============================================================
# STEP 11 — PRINT RESULT
# ============================================================

print("\nFINAL ENTERPRISE STRATEGY\n")
print("=" * 60)
print(result["final_strategy"])